In [1]:
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import top_k_accuracy_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, DataCollatorWithPadding

In [2]:
data = pd.read_csv('labels_encoder.csv',sep=',')
df_test = pd.read_csv('../Dane/test_set.csv')
df_test["text"] = df_test["opis"].astype(str)
test_dataset = Dataset.from_pandas(
    df_test[["text", "labels"]]
)

In [3]:
MODEL_PATH = "./final_model"
MAX_LEN = 128
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
trainer = Trainer(model=model,data_collator=DataCollatorWithPadding(tokenizer))
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LEN,
        padding=False
    )

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [4]:
NUM_LABELS = len(data['labels'].unique())
test_dataset = test_dataset.map(
    tokenize,
    batched=True,
    remove_columns=["text"]
)
test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

Map:   0%|          | 0/7929 [00:00<?, ? examples/s]

### Top accurency 

In [5]:
# prediction output
pred_output = trainer.predict(test_dataset)
y_pred = np.argmax(pred_output.predictions, axis=-1)
y_true = pred_output.label_ids

c:\Badania\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [6]:
probs = torch.softmax(
    torch.tensor(pred_output.predictions),
    dim=-1
).numpy()

top3 = top_k_accuracy_score(
    y_true,
    probs,
    k=3,
    labels=np.arange(NUM_LABELS)
)

top5 = top_k_accuracy_score(
    y_true,
    probs,
    k=5,
    labels=np.arange(NUM_LABELS)
)
top10 = top_k_accuracy_score(
    y_true,
    probs,
    k=10,
    labels=np.arange(NUM_LABELS)
)

print(top3, top5, top10)

0.8085508891411275 0.8518098120822298 0.8968344053474587


### Pojedyńcze predykcja

In [7]:
text = "projektowanie oraz budowa łodzi podwodnych"

In [8]:
inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
outputs = model(**inputs)
y_pred = np.argmax(outputs.logits.detach().numpy(), axis=-1)
print(y_pred)

[228]


In [9]:
data = pd.read_csv('labels_encoder.csv',sep=',')
data.loc[data['labels']==y_pred[0]].head(1)

,id,kod_pkd,opis,labels
80,81,3012,branża: motoryzacyjna | produkty: barracuda 46...,228
